<a href="https://colab.research.google.com/github/nmwiley808/Password-Audit-Tool/blob/main/Password_Audit_Tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Password Audit Tool - IT Project & Cybersecurity Project
### A Python security notebook for auditing password strength, detecting breach exposure, and identifying dangerous patterns

---

## What this project is

This notebook implements a **command-line-ready password auditing pipeline** built for security analysis and portfolio demonstration. It combines multiple layers of password evaluation into a single, unified workflow — from entropy scoring to real-world breach data.

## What it does

| Feature | Tool Used | Description |
|---|---|---|
| **Strength scoring** | `zxcvbn` | Estimates crack time and entropy using Dropbox's realistic model |
| **Breach detection** | HaveIBeenPwned API | k-anonymity SHA1 lookup — your password never leaves your machine |
| **Pattern detection** | `re` (regex) | Flags keyboard walks, dates, repeated chars, l33tspeak |
| **Color-coded report** | `rich` | Green/amber/red table output in the terminal |
| **CSV export** | `csv` + Colab | Download results as a shareable spreadsheet |
| **CLI script** | `click` | Export as `audit.py` — a GitHub-ready command-line tool |

## Who this is for

- Security-minded developers who want to audit password lists
- Students learning applied Python, APIs, and CLI tool design
- Portfolio projects demonstrating real-world Python engineering

## Tech stack

`Python 3` · `zxcvbn` · `requests` · `hashlib` · `rich` · `click` · `re` · `Google Colab`

---
>  **Privacy note:** This tool uses [k-anonymity](https://haveibeenpwned.com/API/v3#SearchingPwnedPasswordsByRange) when checking breach databases. Only the first 5 characters of a SHA1 hash are ever sent to the API — your actual password is never transmitted.

In [ ]:
# Install & Imports
# Install third-party libraries (not in Colab by default)
!pip install zxcvbn rich requests click --quiet

# Standard library
import re
import hashlib
import csv
import io
import os

# Third-party
import requests
import zxcvbn as zx
import click

from rich.console import Console
from rich.table   import Table
from rich         import box as rich_box

# Colab utility (for CSV download in Cell 8) ────
from google.colab import files

# Shared console instance (used across all cells)
console = Console()

result = zx.zxcvbn("test123")

console.print("[bold green]✓ All imports successful[/bold green]")
console.print(f"  zxcvbn smoke test → score: [cyan]{result['score']}[/cyan] "
              f"| crack time: [cyan]{result['crack_times_display']['offline_slow_hashing_1e4_per_second']}[/cyan]")

✓ All imports successful

zxcvbn smoke test → score: 0 | crack time: less than a second

In [ ]:
# Cell 2 — Strength Scorer
# Wraps zxcvbn to return a clean, normalized result dict.
# Score meanings: 0=terrible, 1=bad, 2=weak, 3=good, 4=strong

SCORE_LABELS = {
    0: ("Terrible",  "bold red"),
    1: ("Bad",       "red"),
    2: ("Weak",      "yellow"),
    3: ("Good",      "green"),
    4: ("Strong",    "bold green"),
}

def score_password(password: str) -> dict:
    """
    Run zxcvbn on a password and return a clean result dict.

    Returns:
        score        (int)  : 0–4 strength rating
        label        (str)  : human-readable label ("Weak", "Strong", etc.)
        crack_time   (str)  : estimated offline crack time (display string)
        warning      (str)  : zxcvbn warning, if any
        suggestions  (list) : list of improvement suggestions
        guesses      (int)  : estimated guesses to crack
    """
    r = zx.zxcvbn(password)

    score = r["score"]
    label, _ = SCORE_LABELS[score]

    return {
        "score":       score,
        "label":       label,
        "crack_time":  r["crack_times_display"]["offline_slow_hashing_1e4_per_second"],
        "warning":     r["feedback"]["warning"] or "",
        "suggestions": r["feedback"]["suggestions"],
        "guesses":     r["guesses"],
    }


# Quick test
test_passwords = ["password", "Tr0ub4dor&3", "Xx_Bold_xX", "X9#mK2$pLqR7"]

console.print("\n[bold]Strength Scorer — test run[/bold]")
for pw in test_passwords:
    result = score_password(pw)
    label, color = SCORE_LABELS[result["score"]]
    console.print(
        f"  [{color}]{label:>8}[/{color}]  "
        f"score=[cyan]{result['score']}[/cyan]  "
        f"crack=[cyan]{result['crack_time']}[/cyan]  "
        f"→ [dim]{pw}[/dim]"
    )

Strength Scorer — test run

Terrible  score=0  crack=less than a second  → password

  Strong  score=4  crack=4 months  → Tr0ub4dor&3

    Good  score=3  crack=12 days  → Xx_Bold_xX

  Strong  score=4  crack=3 years  → X9#mK2$pLqR7

In [ ]:
# HIBP Breach Check
# Uses k-anonymity: only the first 5 chars of the SHA1 hash are sent.
# The full password NEVER leaves your matches

HIBP_API = "https://api.pwnedpasswords.com/range/"

def check_breach(password: str) -> int:
    """
    Check HaveIBeenPwned for breach count using k-anonymity

    Hashes the password with SHA1, sends only the first 5 hex chars to the API, then seacrches the returned suffix list locally.

    Returns:
      int: number of times this password appeared in known breaches (0 = clean)
    """

    sha1 = hashlib.sha1(password.encode("utf-8")).hexdigest().upper()
    prefix = sha1[:5]
    suffix = sha1[5:]

    try:
        response = requests.get(HIBP_API + prefix, timeout=5)
        response.raise_for_status()
    except requests.RequestException as e:
        console.print(f"[yellow]⚠ HIBP API unavailable: {e}[/yellow]")
        return -1  # -1 = unknown (API error)

    # Each line is "SUFFIX:COUNT"
    for line in response.text.splitlines():
        parts = line.split(":")
        if len(parts) == 2 and parts[0] == suffix:
            return int(parts[1])

    return 0

# Quick test
console.print("\n[bold]HIBP Breach Check — test run[/bold]")
breach_tests = ["password", "123456", "Xx_KingBob_xX", "X9#mK2$pLqR7!ZzW"]

for pw in breach_tests:
    count = check_breach(pw)
    if count > 0:
        status = f"[bold red]BREACHED {count:,}×[/bold red]"
    elif count == 0:
        status = "[bold green]Clean[/bold green]"
    else:
        status = "[yellow]Unknown (API error)[/yellow]"
    console.print(f"  {status:>30}  → [dim]{pw}[/dim]")

HIBP Breach Check — test run

BREACHED 52,256,179×  → password

BREACHED 209,972,844×  → 123456

Clean  → Xx_KingBob_xX

Clean  → X9#mK2$pLqR7!ZzW

In [ ]:
# Pattern Detector
# Flags dangerous structural patterns that zxcvbn may not fully penalize.
# Returns a list of human-readable flag strings.

# Keyboard walks — common horizontal sequences on a QWERTY layout
KEYBOARD_WALKS = [
    "qwerty", "qwert", "werty", "asdfg", "asdf", "sdfgh",
    "zxcvb", "zxcv", "qazwsx", "12345", "23456", "34567",
    "45678", "56789", "67890", "09876", "98765",
]

# Common l33tspeak substitution map (letter → possible substitutions)
LEET_MAP = {
    "a": ["4", "@"],
    "e": ["3"],
    "i": ["1", "!"],
    "o": ["0"],
    "s": ["5", "$"],
    "t": ["7"],
    "l": ["1"],
    "g": ["9"],
    "b": ["8"],
}

def _reverse_leet(password: str) -> str:
    """Reverse common l33tspeak substitutions to plain text for analysis."""
    p = password.lower()
    for letter, subs in LEET_MAP.items():
        for sub in subs:
            p = p.replace(sub, letter)
    return p

def detect_patterns(password: str) -> list[str]:
    """
    Scan a password for known weak structural patterns.

    Checks for:
      - Keyboard walks (qwerty, asdf, 12345, etc.)
      - Date patterns (DDMMYYYY, MM/DD/YY, YYYY, standalone 4-digit years)
      - Repeated characters (aaa, 111, etc.)
      - L33tspeak substitutions masking a simple word/walk

    Returns:
        list[str]: list of flag descriptions (empty = no patterns found)
    """
    flags = []
    lower = password.lower()
    reversed_leet = _reverse_leet(password)

    # 1. Keyboard walks (direct + reversed)
    for walk in KEYBOARD_WALKS:
        if walk in lower or walk in lower[::-1]:
            flags.append(f"keyboard walk ({walk})")
            break

    # 2. Keyboard walk hidden by l33tspeak
    for walk in KEYBOARD_WALKS:
        if walk in reversed_leet and f"keyboard walk ({walk})" not in flags:
            flags.append(f"l33tspeak keyboard walk ({walk})")
            break

    # 3. Date patterns
    date_patterns = [
        (r"\b(0[1-9]|[12]\d|3[01])(0[1-9]|1[0-2])\d{4}\b", "date (DDMMYYYY)"),
        (r"\b(0[1-9]|1[0-2])[\/\-](0[1-9]|[12]\d|3[01])[\/\-]\d{2,4}\b", "date (MM/DD/YY)"),
        (r"\b(19|20)\d{2}\b", "year (19xx/20xx)"),
    ]
    for pattern, label in date_patterns:
        if re.search(pattern, password):
            flags.append(label)

    # 4. Repeated characters (3+ of the same char in a row)
    if re.search(r"(.)\1{2,}", password):
        flags.append("repeated characters")

    # 5. L33tspeak substitutions (not already flagged as a walk)
    leet_found = any(
        sub in password
        for subs in LEET_MAP.values()
        for sub in subs
    )
    if leet_found and not any("l33tspeak" in f for f in flags):
        flags.append("l33tspeak substitution")

    return flags


# Quick test
console.print("\n[bold]Pattern Detector — test run[/bold]")
pattern_tests = [
    "qwerty123",
    "P@$$w0rd1990",
    "aaaaaBBBB",
    "MyDog01011985",
    "correct horse battery staple",
    "4$df9#Lm!2Xq",
]

for pw in pattern_tests:
    flags = detect_patterns(pw)
    flag_str = ", ".join(flags) if flags else "[green]none[/green]"
    console.print(f"  [dim]{pw:<30}[/dim] → {flag_str}")

Pattern Detector — test run

qwerty123                      → keyboard walk (qwerty), l33tspeak keyboard walk (qwert)

P@$$w0rd1990                   → l33tspeak substitution

aaaaaBBBB                      → repeated characters

MyDog01011985                  → l33tspeak substitution

correct horse battery staple   → none

4$df9#Lm!2Xq                   → l33tspeak keyboard walk (asdfg)

In [ ]:
# Single Password Audit
# Combines scorer + breach check + pattern detector into one unified function.
# Returns a clean result dict used by the bulk auditor and report renderer.

def audit_password(password: str, check_hibp: bool = True) -> dict:
    """
    Full audit of a single password.

    Args:
        password   : the password string to audit
        check_hibp : set False to skip the HIBP API call (e.g. offline mode)

    Returns dict with keys:
        password, score, label, crack_time, warning, suggestions,
        breach_count, patterns, risk_level
    """
    strength = score_password(password)
    breach   = check_breach(password) if check_hibp else -1
    patterns = detect_patterns(password)

    # Derive an overall risk level
    if breach > 0 or strength["score"] <= 1:
        risk = "HIGH"
    elif strength["score"] == 2 or patterns:
        risk = "MEDIUM"
    else:
        risk = "LOW"

    return {
        "password":     password,
        "score":        strength["score"],
        "label":        strength["label"],
        "crack_time":   strength["crack_time"],
        "warning":      strength["warning"],
        "suggestions":  strength["suggestions"],
        "breach_count": breach,
        "patterns":     patterns,
        "risk_level":   risk,
    }


# Quick test
console.print("\n[bold]Single Password Audit — test run[/bold]")
sample = "P@$$w0rd1990"
result = audit_password(sample)

console.print(f"\n  Password   : [cyan]{result['password']}[/cyan]")
console.print(f"  Score      : {result['score']} ({result['label']})")
console.print(f"  Crack time : {result['crack_time']}")
console.print(f"  Breached   : {result['breach_count']:,}× found" if result['breach_count'] > 0 else "  Breached   : [green]No[/green]")
console.print(f"  Patterns   : {', '.join(result['patterns']) if result['patterns'] else '[green]none[/green]'}")
console.print(f"  Risk level : [bold {'red' if result['risk_level'] == 'HIGH' else 'yellow' if result['risk_level'] == 'MEDIUM' else 'green'}]{result['risk_level']}[/bold {'red' if result['risk_level'] == 'HIGH' else 'yellow' if result['risk_level'] == 'MEDIUM' else 'green'}]")

Single Password Audit — test run

Password   : P@$$w0rd1990

Score      : 1 (Bad)

Crack time : 2 seconds

Breached   : 602× found

Patterns   : l33tspeak substitution

Risk level : HIGH

In [ ]:
# Build File Audit
# Audits a list of passwords or reads from a .txt file (one password per line).
# Returns a list of result dicts (same shape as audit_password output).

def audit_list(passwords: list[str], check_hibp: bool = True) -> list[dict]:
    """
    Audit a list of passwords, showing a progress indicator.

    Args:
        passwords  : list of password strings
        check_hibp : set False to skip HIBP checks (faster, offline)

    Returns:
        list of audit result dicts (one per password)
    """
    results = []
    total = len(passwords)

    for i, pw in enumerate(passwords, 1):
        pw = pw.strip()
        if not pw:
            continue
        console.print(f"  Auditing [{i}/{total}]: [dim]{pw[:20]}{'...' if len(pw) > 20 else ''}[/dim]", end="\r")
        results.append(audit_password(pw, check_hibp=check_hibp))

    console.print(f"  [green]✓ Done — {len(results)} passwords audited.[/green]          ")
    return results


def audit_file(filepath: str, check_hibp: bool = True) -> list[dict]:
    """
    Read passwords from a .txt file (one per line) and audit each.

    Args:
        filepath   : path to a plaintext file with one password per line
        check_hibp : set False to skip HIBP checks

    Returns:
        list of audit result dicts
    """
    if not os.path.exists(filepath):
        console.print(f"[bold red]✗ File not found:[/bold red] {filepath}")
        return []

    with open(filepath, "r", encoding="utf-8") as f:
        passwords = [line.strip() for line in f if line.strip()]

    console.print(f"  Loaded [cyan]{len(passwords)}[/cyan] passwords from [dim]{filepath}[/dim]")
    return audit_list(passwords, check_hibp=check_hibp)


# Quick test
sample_passwords = [
    "password",
    "123456",
    "Tr0ub4dor&3",
    "correct horse battery staple",
    "qwerty2024",
    "X9#mK2$pLqR7!Zw",
    "MyDog01011985",
    "P@$$w0rd",
]

console.print("\n[bold]Bulk Audit — test run[/bold]")
bulk_results = audit_list(sample_passwords)
console.print(f"  Sample result (first entry): score={bulk_results[0]['score']}, risk={bulk_results[0]['risk_level']}")

Bulk Audit — test run

Auditing [1/8]: password

Auditing [2/8]: 123456

Auditing [3/8]: Tr0ub4dor&3

Auditing [4/8]: correct horse batter...

...

Auditing [6/8]: X9#mK2$pLqR7!Zw

Auditing [7/8]: MyDog01011985

Auditing [8/8]: P@$$w0rd

✓ Done — 8 passwords audited.

Sample result (first entry): score=0, risk=HIGH

In [ ]:
# Rich Report Renderer
# Renders a color-coded table: green=strong/clean, amber=weak/patterns, red=breached/terrible.

RISK_COLORS = {
    "HIGH":   "bold red",
    "MEDIUM": "yellow",
    "LOW":    "bold green",
}

def render_report(results: list[dict], show_passwords: bool = True) -> None:
    """
    Print a color-coded Rich table summarizing audit results.

    Args:
        results        : list of audit result dicts from audit_password / audit_list
        show_passwords : set False to mask passwords in the output (show first 2 chars + ***)
    """
    table = Table(
        title="Password Audit Report",
        box=rich_box.ROUNDED,
        show_lines=True,
        header_style="bold cyan",
    )

    table.add_column("Password",     style="dim",        max_width=24)
    table.add_column("Score",        justify="center",   max_width=10)
    table.add_column("Crack Time",   justify="left",     max_width=22)
    table.add_column("Breaches",     justify="center",   max_width=10)
    table.add_column("Patterns",     justify="left",     max_width=28)
    table.add_column("Risk",         justify="center",   max_width=8)

    for r in results:
        score      = r["score"]
        breach     = r["breach_count"]
        risk       = r["risk_level"]
        color      = RISK_COLORS[risk]

        # Password display (optionally masked)
        pw_display = r["password"] if show_passwords else r["password"][:2] + "***"

        # Score cell with inline color
        score_label, score_color = SCORE_LABELS[score]
        score_str  = f"[{score_color}]{score}/4\n{score_label}[/{score_color}]"

        # Breach cell
        if breach < 0:
            breach_str = "[yellow]N/A[/yellow]"
        elif breach == 0:
            breach_str = "[bold green]Clean[/bold green]"
        else:
            breach_str = f"[bold red]{breach:,}×[/bold red]"

        # Patterns cell
        if r["patterns"]:
            pattern_str = "[yellow]" + "\n".join(r["patterns"]) + "[/yellow]"
        else:
            pattern_str = "[green]none[/green]"

        # Risk cell
        risk_str = f"[{color}]{risk}[/{color}]"

        table.add_row(pw_display, score_str, r["crack_time"], breach_str, pattern_str, risk_str)

    console.print(table)

    # Summary footer
    high   = sum(1 for r in results if r["risk_level"] == "HIGH")
    medium = sum(1 for r in results if r["risk_level"] == "MEDIUM")
    low    = sum(1 for r in results if r["risk_level"] == "LOW")

    console.print(
        f"\n  [bold]Summary:[/bold]  "
        f"[bold red]{high} HIGH[/bold red]  ·  "
        f"[yellow]{medium} MEDIUM[/yellow]  ·  "
        f"[bold green]{low} LOW[/bold green]  "
        f"({len(results)} total)"
    )


# Quick test
console.print("\n[bold]Rich Report — test run[/bold]\n")
render_report(bulk_results)

Rich Report — test run

                                              Password Audit Report                                              
╭──────────────────────────┬──────────┬────────────────────┬────────────┬──────────────────────────────┬────────╮
│ Password                 │  Score   │ Crack Time         │  Breaches  │ Patterns                     │  Risk  │
├──────────────────────────┼──────────┼────────────────────┼────────────┼──────────────────────────────┼────────┤
│ password                 │   0/4    │ less than a second │ 52,256,17… │ none                         │  HIGH  │
│                          │ Terrible │                    │            │                              │        │
├──────────────────────────┼──────────┼────────────────────┼────────────┼──────────────────────────────┼────────┤
│ 123456                   │   0/4    │ less than a second │ 209,972,8… │ keyboard walk (12345)        │  HIGH  │
│                          │ Terrible │                    │            │ l33tspeak substitution       │        │
├──────────────────────────┼──────────┼────────────────────┼────────────┼──────────────────────────────┼────────┤
│ Tr0ub4dor&3              │   4/4    │ 4 months           │   3,189×   │ l33tspeak substitution       │  HIGH  │
│                          │  Strong  │                    │            │                              │        │
├──────────────────────────┼──────────┼────────────────────┼────────────┼──────────────────────────────┼────────┤
│ correct horse battery    │   4/4    │ centuries          │    387×    │ none                         │  HIGH  │
│ staple                   │  Strong  │                    │            │                              │        │
├──────────────────────────┼──────────┼────────────────────┼────────────┼──────────────────────────────┼────────┤
│ qwerty2024               │   1/4    │ 1 minute           │   3,348×   │ keyboard walk (qwerty)       │  HIGH  │
│                          │   Bad    │                    │            │ l33tspeak keyboard walk      │        │
│                          │          │                    │            │ (qwert)                      │        │
├──────────────────────────┼──────────┼────────────────────┼────────────┼──────────────────────────────┼────────┤
│ X9#mK2$pLqR7!Zw          │   4/4    │ centuries          │   Clean    │ l33tspeak substitution       │ MEDIUM │
│                          │  Strong  │                    │            │                              │        │
├──────────────────────────┼──────────┼────────────────────┼────────────┼──────────────────────────────┼────────┤
│ MyDog01011985            │   3/4    │ 10 hours           │   Clean    │ l33tspeak substitution       │ MEDIUM │
│                          │   Good   │                    │            │                              │        │
├──────────────────────────┼──────────┼────────────────────┼────────────┼──────────────────────────────┼────────┤
│ P@$$w0rd                 │   0/4    │ less than a second │  775,659×  │ l33tspeak substitution       │  HIGH  │
│                          │ Terrible │                    │            │                              │        │
╰──────────────────────────┴──────────┴────────────────────┴────────────┴──────────────────────────────┴────────╯

Summary:  6 HIGH  ·  2 MEDIUM  ·  0 LOW  (8 total)

In [ ]:
# CSV Export
# Saves the audit results to a .csv and triggers a Colab download.

def export_csv(results: list[dict], filename: str = "password_audit_report.csv") -> str:
    """
    Write audit results to a CSV file and trigger a Colab download.

    Columns: password, score, label, crack_time, breach_count, patterns, risk_level, warning

    Args:
        results  : list of audit result dicts
        filename : output filename

    Returns:
        str: path to the written file
    """
    fieldnames = [
        "password", "score", "label", "crack_time",
        "breach_count", "patterns", "risk_level", "warning"
    ]

    with open(filename, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()

        for r in results:
            writer.writerow({
                "password":     r["password"],
                "score":        r["score"],
                "label":        r["label"],
                "crack_time":   r["crack_time"],
                "breach_count": r["breach_count"],
                "patterns":     "; ".join(r["patterns"]) if r["patterns"] else "",
                "risk_level":   r["risk_level"],
                "warning":      r["warning"],
            })

    console.print(f"  [bold green]✓ Saved:[/bold green] [cyan]{filename}[/cyan] ({len(results)} rows)")

    # Trigger download in Colab
    files.download(filename)
    console.print(f"  [bold green]✓ Download triggered.[/bold green]")

    return filename


# Quick test
console.print("\n[bold]CSV Export — test run[/bold]")
export_csv(bulk_results)

CSV Export — test run

✓ Saved: password_audit_report.csv (8 rows)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Download triggered.

'password_audit_report.csv'

In [ ]:
# CLI Export
# Writes the full audit logic as a self-contained audit.py with click CLI commands.
# Run locally with: python audit.py check "mypassword"
#                   python audit.py bulk passwords.txt

CLI_SCRIPT = '''#!/usr/bin/env python3
"""
audit.py — Password Audit CLI Tool
-----------------------------------
A command-line tool to audit passwords for strength, breach exposure,
and dangerous structural patterns.

Usage:
    python audit.py check "mypassword"
    python audit.py bulk passwords.txt
    python audit.py bulk passwords.txt --no-hibp --output report.csv

Requirements:
    pip install zxcvbn rich requests click
"""

import re
import hashlib
import csv
import os
import sys

import click
import requests
import zxcvbn as zx
from rich.console import Console
from rich.table import Table
from rich import box as rich_box

console = Console()

SCORE_LABELS = {
    0: ("Terrible", "bold red"),
    1: ("Bad",      "red"),
    2: ("Weak",     "yellow"),
    3: ("Good",     "green"),
    4: ("Strong",   "bold green"),
}

RISK_COLORS = {"HIGH": "bold red", "MEDIUM": "yellow", "LOW": "bold green"}
HIBP_API    = "https://api.pwnedpasswords.com/range/"

KEYBOARD_WALKS = [
    "qwerty","qwert","werty","asdfg","asdf","sdfgh",
    "zxcvb","zxcv","qazwsx","12345","23456","34567",
    "45678","56789","67890","09876","98765",
]
LEET_MAP = {
    "a":["4","@"],"e":["3"],"i":["1","!"],"o":["0"],
    "s":["5","$"],"t":["7"],"l":["1"],"g":["9"],"b":["8"],
}

def _reverse_leet(password):
    p = password.lower()
    for letter, subs in LEET_MAP.items():
        for sub in subs:
            p = p.replace(sub, letter)
    return p

def score_password(password):
    r = zx.zxcvbn(password)
    score = r["score"]
    label, _ = SCORE_LABELS[score]
    return {
        "score":      score,
        "label":      label,
        "crack_time": r["crack_times_display"]["offline_slow_hashing_1e4_per_second"],
        "warning":    r["feedback"]["warning"] or "",
        "suggestions":r["feedback"]["suggestions"],
        "guesses":    r["guesses"],
    }

def check_breach(password):
    sha1   = hashlib.sha1(password.encode()).hexdigest().upper()
    prefix, suffix = sha1[:5], sha1[5:]
    try:
        response = requests.get(HIBP_API + prefix, timeout=5)
        response.raise_for_status()
    except requests.RequestException:
        return -1
    for line in response.text.splitlines():
        parts = line.split(":")
        if len(parts) == 2 and parts[0] == suffix:
            return int(parts[1])
    return 0

def detect_patterns(password):
    flags, lower = [], password.lower()
    reversed_leet = _reverse_leet(password)
    for walk in KEYBOARD_WALKS:
        if walk in lower or walk in lower[::-1]:
            flags.append(f"keyboard walk ({walk})")
            break
    for walk in KEYBOARD_WALKS:
        if walk in reversed_leet and f"keyboard walk ({walk})" not in flags:
            flags.append(f"l33tspeak keyboard walk ({walk})")
            break
    for pattern, label in [
        (r"\\b(0[1-9]|[12]\\d|3[01])(0[1-9]|1[0-2])\\d{4}\\b", "date (DDMMYYYY)"),
        (r"\\b(0[1-9]|1[0-2])[\\/\\-](0[1-9]|[12]\\d|3[01])[\\/\\-]\\d{2,4}\\b", "date (MM/DD/YY)"),
        (r"\\b(19|20)\\d{2}\\b", "year (19xx/20xx)"),
    ]:
        if re.search(pattern, password):
            flags.append(label)
    if re.search(r"(.)\\1{2,}", password):
        flags.append("repeated characters")
    leet_found = any(sub in password for subs in LEET_MAP.values() for sub in subs)
    if leet_found and not any("l33tspeak" in f for f in flags):
        flags.append("l33tspeak substitution")
    return flags

def audit_password(password, check_hibp=True):
    strength = score_password(password)
    breach   = check_breach(password) if check_hibp else -1
    patterns = detect_patterns(password)
    if breach > 0 or strength["score"] <= 1:
        risk = "HIGH"
    elif strength["score"] == 2 or patterns:
        risk = "MEDIUM"
    else:
        risk = "LOW"
    return {
        "password": password, "score": strength["score"], "label": strength["label"],
        "crack_time": strength["crack_time"], "warning": strength["warning"],
        "suggestions": strength["suggestions"], "breach_count": breach,
        "patterns": patterns, "risk_level": risk,
    }

def render_report(results):
    table = Table(title="Password Audit Report", box=rich_box.ROUNDED,
                  show_lines=True, header_style="bold cyan")
    table.add_column("Password",   style="dim",      max_width=24)
    table.add_column("Score",      justify="center", max_width=10)
    table.add_column("Crack Time", justify="left",   max_width=22)
    table.add_column("Breaches",   justify="center", max_width=10)
    table.add_column("Patterns",   justify="left",   max_width=28)
    table.add_column("Risk",       justify="center", max_width=8)
    for r in results:
        score_label, score_color = SCORE_LABELS[r["score"]]
        breach = r["breach_count"]
        table.add_row(
            r["password"],
            f"[{score_color}]{r[\'score\']}/4\\n{score_label}[/{score_color}]",
            r["crack_time"],
            f"[bold red]{breach:,}×[/bold red]" if breach > 0
                else ("[yellow]N/A[/yellow]" if breach < 0 else "[bold green]Clean[/bold green]"),
            "[yellow]" + "\\n".join(r["patterns"]) + "[/yellow]" if r["patterns"] else "[green]none[/green]",
            f"[{RISK_COLORS[r[\'risk_level\']]}]{r[\'risk_level\']}[/{RISK_COLORS[r[\'risk_level\']]}]",
        )
    console.print(table)
    high   = sum(1 for r in results if r["risk_level"] == "HIGH")
    medium = sum(1 for r in results if r["risk_level"] == "MEDIUM")
    low    = sum(1 for r in results if r["risk_level"] == "LOW")
    console.print(f"\\n  Summary: [bold red]{high} HIGH[/bold red] · [yellow]{medium} MEDIUM[/yellow] · [bold green]{low} LOW[/bold green] ({len(results)} total)")


@click.group()
def cli():
    """Password Audit Tool — check strength, breaches, and patterns."""
    pass

@cli.command()
@click.argument("password")
@click.option("--no-hibp", is_flag=True, default=False, help="Skip HaveIBeenPwned check")
def check(password, no_hibp):
    """Audit a single password."""
    result = audit_password(password, check_hibp=not no_hibp)
    render_report([result])
    if result["suggestions"]:
        console.print("\\n  [bold]Suggestions:[/bold]")
        for s in result["suggestions"]:
            console.print(f"    • {s}")

@cli.command()
@click.argument("filepath", type=click.Path(exists=True))
@click.option("--no-hibp",  is_flag=True, default=False,             help="Skip HIBP checks (faster)")
@click.option("--output",   default="audit_report.csv", show_default=True, help="CSV output filename")
def bulk(filepath, no_hibp, output):
    """Audit all passwords in a .txt file (one per line)."""
    with open(filepath, "r", encoding="utf-8") as f:
        passwords = [l.strip() for l in f if l.strip()]
    console.print(f"  Loaded [cyan]{len(passwords)}[/cyan] passwords from [dim]{filepath}[/dim]")
    results = []
    for i, pw in enumerate(passwords, 1):
        console.print(f"  Auditing [{i}/{len(passwords)}]...", end="\\r")
        results.append(audit_password(pw, check_hibp=not no_hibp))
    console.print(f"  [green]✓ Done[/green]                    ")
    render_report(results)
    with open(output, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=[
            "password","score","label","crack_time","breach_count","patterns","risk_level","warning"
        ])
        writer.writeheader()
        for r in results:
            writer.writerow({**r, "patterns": "; ".join(r["patterns"]), "suggestions": ""})
    console.print(f"  [bold green]✓ Report saved:[/bold green] [cyan]{output}[/cyan]")

if __name__ == "__main__":
    cli()
'''

# Write the script to disk
output_path = "audit.py"
with open(output_path, "w", encoding="utf-8") as f:
    f.write(CLI_SCRIPT)

console.print(f"\n[bold green]✓ audit.py written[/bold green] ({len(CLI_SCRIPT):,} chars)")
console.print("\n  [bold]Run locally with:[/bold]")
console.print("    [cyan]python audit.py check \"mypassword\"[/cyan]")
console.print("    [cyan]python audit.py bulk passwords.txt[/cyan]")
console.print("    [cyan]python audit.py bulk passwords.txt --no-hibp --output report.csv[/cyan]")

# Trigger download so you can save audit.py directly from Colab
files.download(output_path)
console.print("\n  [bold green]✓ audit.py download triggered — save it to your GitHub repo![/bold green]")

✓ audit.py written (7,613 chars)

Run locally with:

python audit.py check "mypassword"

python audit.py bulk passwords.txt

python audit.py bulk passwords.txt --no-hibp --output report.csv

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ audit.py download triggered — save it to your GitHub repo!